# T20 Score Prediction — Fine-tuned Model Evaluation (v2)

## Testing `ratnayakatilanka/t20_score_prediction_keggle-2026-04-18_04.16.40-lite` (LoRA fine-tuned LLaMA 3.2-3B)

Evaluating the fine-tuned model at the **minimum eval-loss checkpoint** (step 800, eval/loss = 1.85624)
on the held-out test split of our T20 first-innings score prediction dataset.

The base model (`meta-llama/Llama-3.2-3B`) is loaded with 4-bit quantization and the LoRA adapters
are applied on top using a specific revision SHA that corresponds to the best checkpoint.

**LITE_MODE = True** → T4 GPU (free Colab), 4-bit QLoRA, evaluate 200 datapoints

In [ ]:
!pip install -q --upgrade bitsandbytes==0.48.2 peft accelerate transformers datasets
!wget -q https://raw.githubusercontent.com/ratnayakatilanka/t20_score_prediction/main/research/util.py -O util.py

In [ ]:
import os
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import PeftModel
from datasets import load_dataset
from util import evaluate

In [ ]:
# ── Constants ────────────────────────────────────────────────────────────────
BASE_MODEL       = "meta-llama/Llama-3.2-3B"
HF_USER          = "ratnayakatilanka"
PROJECT_NAME     = "t20_score_prediction"
FINETUNED_MODEL  = "ratnayakatilanka/t20_score_prediction_keggle-2026-04-18_04.16.40-lite"
REVISION         = "00be3f3f2cb4af3cd727c68284ca3dcbad7656a4"   # step-800, min eval/loss = 1.85624
DATASET_NAME     = f"{HF_USER}/t20_score_prediction_prompts"

LITE_MODE        = True   # True → T4 (free Colab)

# ── Quantization / precision ──────────────────────────────────────────────────
QUANT_4_BIT      = True
use_bf16         = torch.cuda.is_bf16_supported()   # False on T4, True on A100

# ── Evaluation size ───────────────────────────────────────────────────────────
EVAL_SIZE        = 200 if LITE_MODE else 500

print(f"LITE_MODE        : {LITE_MODE}")
print(f"Base model       : {BASE_MODEL}")
print(f"Fine-tuned model : {FINETUNED_MODEL}")
print(f"Revision (step-800): {REVISION}")
print(f"QUANT_4_BIT      : {QUANT_4_BIT}")
print(f"bfloat16         : {use_bf16}")
print(f"EVAL_SIZE        : {EVAL_SIZE}")

In [ ]:
# Log in to Hugging Face (the fine-tuned model repo is private)
hf_token = userdata.get("HF_TOKEN")
login(hf_token, add_to_git_credential=True)
print("Logged in to Hugging Face")

## Load Dataset

In [ ]:
dataset = load_dataset(DATASET_NAME)
test_data = list(dataset["test"])

print(dataset)
print(f"\nTest set size: {len(test_data):,} rows  (evaluating {EVAL_SIZE})")
print("\n--- Sample datapoint ---")
print(test_data[0]["prompt"])
print("\n--- Completion ---", test_data[0]["completion"])

## Load Tokenizer and Fine-tuned Model

The base model is loaded with **4-bit quantization** (suitable for T4), then the LoRA adapters
from the specific revision (step-800, minimum eval/loss checkpoint) are applied on top using
`PeftModel.from_pretrained`.

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load base model with 4-bit quantization
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)

# Apply LoRA adapters from the specific revision (step-800 checkpoint)
fine_tuned_model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL, revision=REVISION)
fine_tuned_model.eval()
print(f"Base model       : {BASE_MODEL}")
print(f"Adapters applied : {FINETUNED_MODEL}")
print(f"Revision         : {REVISION}")

## Evaluate Fine-tuned Model

In [ ]:
def model_predict(item):
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

In [ ]:
set_seed(42)
evaluate(model_predict, test_data)

## Results

### 200 datapoints

| Model | Avg Error (runs) | MSE | r² |
|---|---|---|---|
| Revision (step-800, eval/loss = 1.85624) | 17.05 | 601 | 66.4% |
| Full (final checkpoint) | **16.12** | 618 | 65.4% |

### 500 datapoints

| Model | Avg Error (runs) | MSE | r² |
|---|---|---|---|
| **Revision** (step-800, eval/loss = 1.85624) | **16.51** | **543** | **68.3%** |
| Full (final checkpoint) | 16.54 | 546 | 68.1% |

The 200-datapoint result showing the full model with better avg error (16.12 vs 17.05) was
attributable to sample noise from the chronologically-ordered test set. At 500 datapoints
the revision model wins on all three metrics.

**Conclusion:** The minimum eval/loss checkpoint (`00be3f3f`) is the better model and the
recommended choice for deployment.